In [ ]:
import os
import shutil
from google.colab import drive

# 1. Flush and unmount any dangling or stale mount points
try:
    drive.flush_and_unmount()
except Exception:
    pass

# 2. Remove lingering non-mount directory to avoid 'Mountpoint must not already contain files'
mount_point = '/content/drive'
if os.path.exists(mount_point):
    if not os.path.ismount(mount_point):
        shutil.rmtree(mount_point)

# 3. Clean force remount
drive.mount(mount_point, force_remount=True)

In [ ]:
"""
Select Best-Performing Run per (Content, Style, Architecture) Triad
Ranks shared image pairs based on the optimal run (highest SSIM + (1 - LPIPS))
without averaging across repeated runs.
"""
import pandas as pd

# ==========================================
# 1. LOAD DATASET & COMPUTE COMPOUND SCORE
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets/Output_subset"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "master_unified_metrics.csv")
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not locate master evaluation CSV. Verify file path.")

df = pd.read_csv(csv_path)

# Enforce clean strings and numeric types
df["Content_Image"] = df["Content_Image"].astype(str).str.lower().str.strip()
df["Style_Image"] = df["Style_Image"].astype(str).str.lower().str.strip()
df["SSIM_Content"] = pd.to_numeric(df["SSIM_Content"], errors="coerce")
df["LPIPS_Content"] = pd.to_numeric(df["LPIPS_Content"], errors="coerce")
df["Isolated_Gram_Style"] = pd.to_numeric(df["Isolated_Gram_Style"], errors="coerce")

# Calculate compound score: SSIM + (1 - LPIPS)
# High SSIM and Low LPIPS both maximize this value
df["Compound_Score"] = df["SSIM_Content"] + (1.0 - df["LPIPS_Content"])

# Drop entries missing evaluation scores
df = df.dropna(subset=["SSIM_Content", "LPIPS_Content"]).copy()

# ==========================================
# 2. SELECT BEST RUN PER PAIR & ARCHITECTURE
# ==========================================
# Sort descending by Compound_Score and pick the top run (no averaging)
best_runs_df = (
    df.sort_values(by="Compound_Score", ascending=False)
    .groupby(["Content_Image", "Style_Image", "Architecture"], as_index=False)
    .first()
)

# Standardize architecture labels for column prefixes
arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN",
}
best_runs_df["Arch_Tag"] = best_runs_df["Architecture"].map(arch_map).fillna("Unknown")

# ==========================================
# 3. PIVOT TABLE TO ALIGN IDENTICAL INPUT PAIRS
# ==========================================
pivoted = best_runs_df.pivot(
    index=["Content_Image", "Style_Image"],
    columns="Arch_Tag",
    values=[
        "Output_Image",
        "Run_Index",
        "SSIM_Content",
        "LPIPS_Content",
        "Isolated_Gram_Style",
        "Compound_Score",
    ],
)

# Flatten hierarchical columns: e.g. "Output_Image_Gatys", "Compound_Score_AdaIN"
pivoted.columns = [f"{metric}_{arch}" for metric, arch in pivoted.columns]
pivoted = pivoted.reset_index()

# Retain only input pairs that succeeded across all three architectures
required_score_cols = [
    "Compound_Score_Gatys",
    "Compound_Score_FastNST",
    "Compound_Score_AdaIN",
]
pivoted = pivoted.dropna(subset=required_score_cols).copy()

# Pair-level utility: Mean compound performance of the best runs
pivoted["Pair_Mean_Compound"] = pivoted[required_score_cols].mean(axis=1)

# Pair-level divergence: Max difference in compound score between architectures
pivoted["Arch_Compound_Divergence"] = (
    pivoted[required_score_cols].max(axis=1) - pivoted[required_score_cols].min(axis=1)
)

OUTPUT_DIR = os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "Best_Run_Stratification")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 4. EXTRACT BEST & WORST INPUT PAIRS
# ==========================================
# Best Pairs for Human Evaluation (Highest aggregate compound score across best runs)
best_pairs = pivoted.sort_values(by="Pair_Mean_Compound", ascending=False).reset_index(drop=True)
best_pairs.to_csv(os.path.join(OUTPUT_DIR, "best_pairs_best_run_human_eval.csv"), index=False)

# Worst Pairs for Report Diagnosis (Lowest aggregate compound score across best runs)
worst_pairs = pivoted.sort_values(by="Pair_Mean_Compound", ascending=True).reset_index(drop=True)
worst_pairs.to_csv(os.path.join(OUTPUT_DIR, "worst_pairs_best_run_diagnosis.csv"), index=False)

# Display Key Columns
display_cols = [
    "Content_Image", "Style_Image",
    "Output_Image_Gatys", "Compound_Score_Gatys",
    "Output_Image_FastNST", "Compound_Score_FastNST",
    "Output_Image_AdaIN", "Compound_Score_AdaIN",
    "Pair_Mean_Compound"
]

print("=" * 120)
print("TOP 5 BEST IMAGE PAIRS (BASED ON BEST SINGLE RUN COMPOUND SCORE)")
print("Recommended for MOS / 2AFC Human Evaluation Study")
print("=" * 120)
print(best_pairs[display_cols].head(10).to_string(index=False))

print("\n" + "=" * 120)
print("TOP 5 WORST IMAGE PAIRS (BASED ON BEST SINGLE RUN COMPOUND SCORE)")
print("Recommended for Section 5.2 Failure Analysis & Saliency Map Diagnosis")
print("=" * 120)
print(worst_pairs[display_cols].head(10).to_string(index=False))

print(f"\n[✓] Extracted full tables successfully to:\n    {OUTPUT_DIR}")


In [ ]:
"""
Balanced Multi-Modal Image Pair Stratification Engine
Incorporates Isolated Gram Style MSE into the compound objective to prevent
under-stylization bias.

Formulation:
  1. Content Score = 0.5 * SSIM + 0.5 * (1 - LPIPS)  [Range: 0 to 1]
  2. Style Score   = MinMax(-log10(Gram_MSE))        [Range: 0 to 1]
  3. Balanced Utility = 0.50 * Content Score + 0.50 * Style Score

Selects the single best run per (Content, Style, Architecture) triad and
ranks shared pairs across all three architectures.
"""

import numpy as np

# ==========================================
# 1. LOAD DATASET & SANITIZE
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets/Output_subset"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "master_unified_metrics.csv")
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not locate master evaluation CSV. Verify file path.")

df = pd.read_csv(csv_path)

# Enforce clean strings and numeric types
df["Content_Image"] = df["Content_Image"].astype(str).str.lower().str.strip()
df["Style_Image"] = df["Style_Image"].astype(str).str.lower().str.strip()
df["SSIM_Content"] = pd.to_numeric(df["SSIM_Content"], errors="coerce")
df["LPIPS_Content"] = pd.to_numeric(df["LPIPS_Content"], errors="coerce")
df["Isolated_Gram_Style"] = pd.to_numeric(df["Isolated_Gram_Style"], errors="coerce")

# Drop incomplete records
df = df.dropna(subset=["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]).copy()

# ==========================================
# 2. LOG-NORMALIZED STYLE SCORE & 50/50 BALANCE
# ==========================================
# Gram MSE spans logarithmic orders (1e-5 to 1e-2).
# Transform via negative log so that lower error yields higher score.
eps = 1e-12
log_gram = -np.log10(df["Isolated_Gram_Style"] + eps)
min_log, max_log = log_gram.min(), log_gram.max()

# Min-Max scale Style Score strictly to [0, 1]
df["Style_Fidelity_Score"] = (log_gram - min_log) / (max_log - min_log + 1e-8)

# Normalized Content Retention Score in [0, 1]
# SSIM measures geometry; (1 - LPIPS) measures deep perceptual layout
df["Content_Retention_Score"] = 0.5 * df["SSIM_Content"] + 0.5 * (1.0 - df["LPIPS_Content"])

# Balanced Compound Utility: 50% Content Preservation vs. 50% Artistic Texture Transfer
df["Balanced_Compound_Score"] = (
    0.50 * df["Content_Retention_Score"] + 0.50 * df["Style_Fidelity_Score"]
)

# ==========================================
# 3. SELECT BEST RUN PER TRIAD (NO RUN AVERAGING)
# ==========================================
best_runs_df = (
    df.sort_values(by="Balanced_Compound_Score", ascending=False)
    .groupby(["Content_Image", "Style_Image", "Architecture"], as_index=False)
    .first()
)

arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN",
}
best_runs_df["Arch_Tag"] = best_runs_df["Architecture"].map(arch_map).fillna("Unknown")

# ==========================================
# 4. PIVOT TABLE ACROSS ARCHITECTURES
# ==========================================
pivoted = best_runs_df.pivot(
    index=["Content_Image", "Style_Image"],
    columns="Arch_Tag",
    values=[
        "Output_Image",
        "Run_Index",
        "SSIM_Content",
        "LPIPS_Content",
        "Isolated_Gram_Style",
        "Content_Retention_Score",
        "Style_Fidelity_Score",
        "Balanced_Compound_Score",
    ],
)

pivoted.columns = [f"{metric}_{arch}" for metric, arch in pivoted.columns]
pivoted = pivoted.reset_index()

required_score_cols = [
    "Balanced_Compound_Score_Gatys",
    "Balanced_Compound_Score_FastNST",
    "Balanced_Compound_Score_AdaIN",
]
pivoted = pivoted.dropna(subset=required_score_cols).copy()

# Cross-architecture aggregate balanced utility
pivoted["Pair_Mean_Balanced"] = pivoted[required_score_cols].mean(axis=1)

# Divergence between highest and lowest performing architecture for this pair
pivoted["Arch_Balanced_Divergence"] = (
    pivoted[required_score_cols].max(axis=1) - pivoted[required_score_cols].min(axis=1)
)

OUTPUT_DIR = os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "Balanced_Run_Stratification")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 5. EXPORT BEST & WORST BALANCED PAIRS
# ==========================================
best_pairs = pivoted.sort_values(by="Pair_Mean_Balanced", ascending=False).reset_index(drop=True)
best_pairs.to_csv(os.path.join(OUTPUT_DIR, "best_balanced_pairs_human_eval.csv"), index=False)

worst_pairs = pivoted.sort_values(by="Pair_Mean_Balanced", ascending=True).reset_index(drop=True)
worst_pairs.to_csv(os.path.join(OUTPUT_DIR, "worst_balanced_pairs_diagnosis.csv"), index=False)

display_cols = [
    "Content_Image", "Style_Image",
    "Output_Image_Gatys", "Balanced_Compound_Score_Gatys",
    "Output_Image_FastNST", "Balanced_Compound_Score_FastNST",
    "Output_Image_AdaIN", "Balanced_Compound_Score_AdaIN",
    "Pair_Mean_Balanced"
]

print("=" * 125)
print("TOP 10 BEST BALANCED PAIRS (50% CONTENT RETENTION + 50% STYLE FIDELITY)")
print("Recommended for Human Evaluation (Genuine Stylization + Preserved Identity)")
print("=" * 125)
print(best_pairs[display_cols].head(10).to_string(index=False))

print("\n" + "=" * 125)
print("TOP 10 WORST BALANCED PAIRS (LOWEST COMBINED SCORE)")
print("Recommended for Section 5.2 Failure Mode Analysis & Saliency Maps")
print("=" * 125)
print(worst_pairs[display_cols].head(10).to_string(index=False))

print(f"\n[✓] Balanced analysis complete. Artifacts saved to:\n    {OUTPUT_DIR}")

In [ ]:
"""
Interactive NST Tripartite Explorer for Google Colab / Jupyter
Provides dynamic sliders to adjust the Content vs. Style weighting
and set a hard minimum style threshold to eliminate under-stylized artifacts.
"""
import ipywidgets as widgets
from IPython.display import display, HTML

# ==========================================
# 1. LOAD & PRE-PROCESS DATASET
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets/Output_subset"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "master_unified_metrics.csv"),
    os.path.join(BASE_DIR, "Benchmark_Analysis", "master_unified_evaluations.csv"),
    os.path.join(BASE_DIR, "master_unified_metrics.csv"),
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not find master_unified_metrics.csv. Check directory path.")

raw_df = pd.read_csv(csv_path)

# Sanitize strings & coerce numerics
raw_df["Content_Image"] = raw_df["Content_Image"].astype(str).str.lower().str.strip()
raw_df["Style_Image"] = raw_df["Style_Image"].astype(str).str.lower().str.strip()
raw_df["SSIM_Content"] = pd.to_numeric(raw_df["SSIM_Content"], errors="coerce")
raw_df["LPIPS_Content"] = pd.to_numeric(raw_df["LPIPS_Content"], errors="coerce")
raw_df["Isolated_Gram_Style"] = pd.to_numeric(raw_df["Isolated_Gram_Style"], errors="coerce")

df_clean = raw_df.dropna(subset=["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]).copy()

# Precompute standard normalized scores in [0, 1]
# 1. Content Score: Geometric retention (SSIM) + Perceptual identity (1 - LPIPS)
df_clean["Score_Content"] = 0.5 * df_clean["SSIM_Content"] + 0.5 * (1.0 - df_clean["LPIPS_Content"])

# 2. Style Score: Log-linearized Gram MSE (Lower error -> Higher score)
log_gram = -np.log10(df_clean["Isolated_Gram_Style"] + 1e-12)
min_log, max_log = log_gram.min(), log_gram.max()
df_clean["Score_Style"] = (log_gram - min_log) / (max_log - min_log + 1e-8)

arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN",
}
df_clean["Arch_Tag"] = df_clean["Architecture"].map(arch_map).fillna("Unknown")

# ==========================================
# 2. INTERACTIVE RANKING FUNCTION
# ==========================================
out_display = widgets.Output()

def evaluate_pairs(style_weight, enable_min_style, min_style_threshold, top_n):
    out_display.clear_output()
    content_weight = 1.0 - style_weight

    with out_display:
        working_df = df_clean.copy()

        # Optional: Filter out outputs below minimum style threshold
        if enable_min_style:
            working_df = working_df[working_df["Score_Style"] >= min_style_threshold]
            if working_df.empty:
                print("No samples meet the minimum style threshold. Lower the threshold slider.")
                return

        # Dynamic utility: w_c * Content + w_s * Style
        working_df["Dynamic_Utility"] = (
            content_weight * working_df["Score_Content"] +
            style_weight * working_df["Score_Style"]
        )

        # Select the single best-performing run per (Content, Style, Architecture) triad
        best_runs = (
            working_df.sort_values(by="Dynamic_Utility", ascending=False)
            .groupby(["Content_Image", "Style_Image", "Arch_Tag"], as_index=False)
            .first()
        )

        # Pivot to compare architectures across identical pairs
        pivoted = best_runs.pivot(
            index=["Content_Image", "Style_Image"],
            columns="Arch_Tag",
            values=["Dynamic_Utility", "Score_Content", "Score_Style", "Output_Image"]
        )
        pivoted.columns = [f"{metric}_{arch}" for metric, arch in pivoted.columns]
        pivoted = pivoted.reset_index()

        required_cols = [c for c in pivoted.columns if c.startswith("Dynamic_Utility_")]

        # Require presence in all available models after filtering
        pivoted = pivoted.dropna(subset=required_cols).copy()
        if pivoted.empty:
            print("No shared pairs survived across all three architectures under these thresholds.")
            return

        pivoted["Pair_Mean_Utility"] = pivoted[required_cols].mean(axis=1)

        # Sort best and worst
        ranked_best = pivoted.sort_values(by="Pair_Mean_Utility", ascending=False).head(top_n)
        ranked_worst = pivoted.sort_values(by="Pair_Mean_Utility", ascending=True).head(top_n)

        display_cols = [
            "Content_Image", "Style_Image", "Pair_Mean_Utility",
            "Dynamic_Utility_Gatys", "Dynamic_Utility_FastNST", "Dynamic_Utility_AdaIN",
            "Output_Image_Gatys", "Output_Image_FastNST", "Output_Image_AdaIN"
        ]
        available_cols = [c for c in display_cols if c in pivoted.columns]

        print(f"--- [ACTIVE CONFIGURATION] ---")
        print(f"Content Weight: {content_weight:.2f} | Style Weight: {style_weight:.2f}")
        if enable_min_style:
            print(f"Active Filter: Enforcing Score_Style >= {min_style_threshold:.2f}")
        print(f"Total Shared Pairs Available: {len(pivoted)}\n")

        display(HTML(f"<h4>Top {top_n} Best Balanced Pairs (Recommended for Human Survey)</h4>"))
        display(ranked_best[available_cols].style.background_gradient(subset=["Pair_Mean_Utility"], cmap="viridis"))

        display(HTML(f"<h4>Top {top_n} Worst Performing Pairs (Recommended for Section 5.2 Failure Saliency)</h4>"))
        display(ranked_worst[available_cols].style.background_gradient(subset=["Pair_Mean_Utility"], cmap="magma_r"))

# ==========================================
# 3. WIDGET CONTROLS UI
# ==========================================
style_slider = widgets.FloatSlider(
    value=0.50, min=0.0, max=1.0, step=0.05,
    description="Style Weight:",
    tooltip="Higher shifts preference toward intense artistic transfer; Lower shifts toward identity preservation.",
    continuous_update=False,
    layout=widgets.Layout(width="420px")
)

min_style_check = widgets.Checkbox(
    value=False,
    description="Enforce Min Style Cutoff",
    tooltip="Filters out under-stylized results (near-identity outputs)",
    layout=widgets.Layout(width="220px")
)

threshold_slider = widgets.FloatSlider(
    value=0.35, min=0.05, max=0.85, step=0.05,
    description="Min Style:",
    continuous_update=False,
    layout=widgets.Layout(width="380px")
)

top_n_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Top N Pairs:",
    continuous_update=False,
    layout=widgets.Layout(width="300px")
)

# Bind widget controls to callback
ui = widgets.VBox([
    widgets.HBox([style_slider, top_n_slider]),
    widgets.HBox([min_style_check, threshold_slider])
])

interactive_binding = widgets.interactive_output(
    evaluate_pairs,
    {
        "style_weight": style_slider,
        "enable_min_style": min_style_check,
        "min_style_threshold": threshold_slider,
        "top_n": top_n_slider,
    }
)

display(ui, interactive_binding, out_display)

In [ ]:
"""
Multi-Row Decomposed Metric Heatmap Grid
Generates a 3-Row x 3-Column matrix across all architectures:
  - Row 1: Balanced Compound Utility (50% Content + 50% Style)
  - Row 2: Content Retention Score Only (SSIM & 1 - LPIPS)
  - Row 3: Style Fidelity Score Only (Log-Linearized Isolated Gram MSE)
"""

import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. DATA INGESTION & SANITIZATION
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets/Output_subset"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "master_unified_metrics.csv")
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not locate master_unified_metrics.csv.")

df = pd.read_csv(csv_path)

def sanitize_label(val):
    s = str(val).lower().strip()
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        s = s.replace(ext, "")
    s = s.replace("starry_night", "starrynight")
    s = s.replace("composition7", "composition_vii").replace("composition_7", "composition_vii")
    return s

df["Content_Image"] = df["Content_Image"].apply(sanitize_label)
df["Style_Image"] = df["Style_Image"].apply(sanitize_label)

df["SSIM_Content"] = pd.to_numeric(df["SSIM_Content"], errors="coerce")
df["LPIPS_Content"] = pd.to_numeric(df["LPIPS_Content"], errors="coerce")
df["Isolated_Gram_Style"] = pd.to_numeric(df["Isolated_Gram_Style"], errors="coerce")

df = df.dropna(subset=["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]).copy()

# ==========================================
# 2. INDEPENDENT METRIC NORMALIZATION
# ==========================================
# A. Content Retention: Geometric retention (SSIM) + Semantic layout (1 - LPIPS)
df["Content_Score"] = 0.5 * df["SSIM_Content"] + 0.5 * (1.0 - df["LPIPS_Content"])

# B. Style Fidelity: Log-linearized Gram MSE (Lower error -> Higher score)
log_gram = -np.log10(df["Isolated_Gram_Style"] + 1e-12)
df["Style_Score"] = (log_gram - log_gram.min()) / (log_gram.max() - log_gram.min() + 1e-8)

# C. Balanced Compound
df["Balanced_Compound"] = 0.50 * df["Content_Score"] + 0.50 * df["Style_Score"]

# Select the single best run per (Content, Style, Architecture) based on balanced utility
best_runs = (
    df.sort_values(by="Balanced_Compound", ascending=False)
    .groupby(["Content_Image", "Style_Image", "Architecture"], as_index=False)
    .first()
)

detected_archs = df["Architecture"].unique().tolist()
detected_contents = sorted(df["Content_Image"].unique().tolist())
detected_styles = sorted(df["Style_Image"].unique().tolist())

# ==========================================
# 3. CONSTRUCT 3-ROW MULTI-METRIC GRID
# ==========================================
# Define rows configuration
rows_config = [
    {
        "metric": "Balanced_Compound",
        "title_prefix": "Balanced Compound",
        "cmap": "viridis",
        "cbar_label": "Composite Utility [0, 1]"
    },
    {
        "metric": "Content_Score",
        "title_prefix": "Content Only (SSIM + [1-LPIPS])",
        "cmap": "Blues",
        "cbar_label": "Content Preservation [0, 1]"
    },
    {
        "metric": "Style_Score",
        "title_prefix": "Style Only (Isolated Gram MSE)",
        "cmap": "magma",
        "cbar_label": "Style Fidelity [0, 1]"
    }
]

n_rows = len(rows_config)
n_cols = len(detected_archs)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7.5 * n_cols, 5.5 * n_rows), sharex=True, sharey=True)

if n_cols == 1:
    axes = np.expand_dims(axes, axis=1)

for r_idx, r_conf in enumerate(rows_config):
    metric_col = r_conf["metric"]
    vmin = best_runs[metric_col].min()
    vmax = best_runs[metric_col].max()

    for c_idx, arch in enumerate(detected_archs):
        ax = axes[r_idx, c_idx]
        arch_df = best_runs[best_runs["Architecture"] == arch]

        matrix = arch_df.pivot(index="Content_Image", columns="Style_Image", values=metric_col)
        matrix = matrix.reindex(index=detected_contents, columns=detected_styles)

        cbar_active = (c_idx == n_cols - 1)
        sns.heatmap(
            matrix,
            ax=ax,
            cmap=r_conf["cmap"],
            vmin=vmin,
            vmax=vmax,
            annot=True,
            fmt=".2f",
            annot_kws={"size": 9, "weight": "bold"},
            cbar=cbar_active,
            cbar_kws={"label": r_conf["cbar_label"]} if cbar_active else None,
            linewidths=0.5,
            linecolor="white",
            square=True
        )

        # Titles and axis labeling
        if r_idx == 0:
            ax.set_title(f"{arch}\n[{r_conf['title_prefix']}]", fontsize=11, fontweight="bold", pad=10)
        else:
            ax.set_title(f"[{r_conf['title_prefix']}]", fontsize=10, fontweight="bold", pad=8)

        if c_idx == 0:
            ax.set_ylabel(f"{r_conf['title_prefix']}\n\nContent Target", fontsize=10, fontweight="bold")
        else:
            ax.set_ylabel("")

        if r_idx == n_rows - 1:
            ax.set_xlabel("Style Target Image", fontsize=10, fontweight="bold")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
        else:
            ax.set_xlabel("")

plt.suptitle(
    "Cross-Architecture Decomposed Heatmap Analysis\n"
    "Row 1: 50/50 Balanced Utility | Row 2: Content Preservation Only | Row 3: Textural Style Fidelity Only",
    fontsize=14,
    fontweight="bold",
    y=0.995
)

plt.tight_layout()

# Export figure
FIGURES_DIR = os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
save_path = os.path.join(FIGURES_DIR, "fig_decomposed_content_style_heatmaps.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"[✓] Decomposed multi-row heatmap saved to:\n    {save_path}")

In [ ]:
# Check raw values for Gatys
gatys_df = df[df["Architecture"].str.contains("Gatys", case=False, na=False)]
print("Unique Gatys Style strings (raw):", gatys_df["Style_Image"].unique())
print("Gatys Starry Night row count:", gatys_df[gatys_df["Style_Image"].astype(str).str.contains("starry", case=False)].shape[0])
print("NaN count in Gatys Starry Night:",
      gatys_df[gatys_df["Style_Image"].astype(str).str.contains("starry", case=False)][["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]].isna().sum().to_dict())

In [ ]:
"""
Balanced Multi-Modal Stratification with Architecture-Specific Normalization
- Normalizes style fidelity (-log10 Gram MSE) within each architecture's own [min, max].
- Applies a >10th percentile style cutoff to qualify for Best Performing (pruning under-stylization).
- Imposes no threshold for Worst Performing (capturing complete failure modes).
"""
# ==========================================
# 1. LOAD & SANITIZE DATASET
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets/Output_subset"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Comprehensive_Benchmark_Analysis", "master_unified_metrics.csv")
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not find master_unified_metrics.csv.")

df = pd.read_csv(csv_path)

def sanitize_label(val):
    s = str(val).lower().strip()
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        s = s.replace(ext, "")
    if "starry" in s:
        return "starrynight"
    if "composition" in s:
        return "composition_vii"
    if "muse" in s:
        return "la_muse"
    if "feather" in s:
        return "feathers"
    return s.replace(" ", "_").replace("-", "_")

df["Content_Image"] = df["Content_Image"].apply(sanitize_label)
df["Style_Image"] = df["Style_Image"].apply(sanitize_label)

for col in ["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]).copy()

# ==========================================
# 2. ARCHITECTURE-SPECIFIC NORMALIZATION
# ==========================================
# Content Retention Score in [0, 1]
df["Score_Content"] = 0.5 * df["SSIM_Content"] + 0.5 * (1.0 - df["LPIPS_Content"])

# Normalize Style Score strictly within each architecture's min-max bounds
df["Score_Style"] = 0.0
df["P10_Style_Cutoff"] = 0.0

for arch in df["Architecture"].unique():
    arch_mask = df["Architecture"] == arch
    log_gram = -np.log10(df.loc[arch_mask, "Isolated_Gram_Style"] + 1e-12)

    min_val = log_gram.min()
    max_val = log_gram.max()

    # Min-Max normalization within architecture
    norm_style = (log_gram - min_val) / (max_val - min_val + 1e-8)
    df.loc[arch_mask, "Score_Style"] = norm_style

    # 10th percentile cutoff for this specific architecture
    p10 = norm_style.quantile(0.10)
    df.loc[arch_mask, "P10_Style_Cutoff"] = p10

# Balanced Compound Utility: 50% Content + 50% Intra-Architecture Style
df["Balanced_Compound"] = 0.50 * df["Score_Content"] + 0.50 * df["Score_Style"]

arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN",
}
df["Arch_Tag"] = df["Architecture"].map(arch_map).fillna("Unknown")

# ==========================================
# 3. BEST PERFORMING SELECTION (STYLE > 10th PERCENTILE)
# ==========================================
# Prune runs falling in the bottom 10% of stylistic transfer for that model
qualifying_runs = df[df["Score_Style"] > df["P10_Style_Cutoff"]].copy()

best_runs_df = (
    qualifying_runs.sort_values(by="Balanced_Compound", ascending=False)
    .groupby(["Content_Image", "Style_Image", "Arch_Tag"], as_index=False)
    .first()
)

best_pivoted = best_runs_df.pivot(
    index=["Content_Image", "Style_Image"],
    columns="Arch_Tag",
    values=["Balanced_Compound", "Score_Content", "Score_Style", "Output_Image"]
)
best_pivoted.columns = [f"{metric}_{arch}" for metric, arch in best_pivoted.columns]
best_pivoted = best_pivoted.reset_index()

required_best = [c for c in best_pivoted.columns if c.startswith("Balanced_Compound_")]
best_pivoted = best_pivoted.dropna(subset=required_best).copy()
best_pivoted["Pair_Mean_Compound"] = best_pivoted[required_best].mean(axis=1)

ranked_best_pairs = best_pivoted.sort_values(by="Pair_Mean_Compound", ascending=False).reset_index(drop=True)

# ==========================================
# 4. WORST PERFORMING SELECTION (NO FILTER)
# ==========================================
# Full dataset without style floor: identify absolute failure modes
worst_runs_df = (
    df.sort_values(by="Balanced_Compound", ascending=True)
    .groupby(["Content_Image", "Style_Image", "Arch_Tag"], as_index=False)
    .first()
)

worst_pivoted = worst_runs_df.pivot(
    index=["Content_Image", "Style_Image"],
    columns="Arch_Tag",
    values=["Balanced_Compound", "Score_Content", "Score_Style", "Output_Image"]
)
worst_pivoted.columns = [f"{metric}_{arch}" for metric, arch in worst_pivoted.columns]
worst_pivoted = worst_pivoted.reset_index()

required_worst = [c for c in worst_pivoted.columns if c.startswith("Balanced_Compound_")]
worst_pivoted = worst_pivoted.dropna(subset=required_worst).copy()
worst_pivoted["Pair_Mean_Compound"] = worst_pivoted[required_worst].mean(axis=1)

ranked_worst_pairs = worst_pivoted.sort_values(by="Pair_Mean_Compound", ascending=True).reset_index(drop=True)

# ==========================================
# 5. OUTPUT & EXPORT
# ==========================================
display_cols = [
    "Content_Image", "Style_Image",
    "Balanced_Compound_Gatys", "Score_Style_Gatys",
    "Balanced_Compound_FastNST", "Score_Style_FastNST",
    "Balanced_Compound_AdaIN", "Score_Style_AdaIN",
    "Pair_Mean_Compound"
]
available_cols = [c for c in display_cols if c in ranked_best_pairs.columns]

print("=" * 120)
print("QUALIFYING BEST PAIRS (Score_Style > 10th Percentile within Architecture)")
print("Recommended for Human MOS / 2AFC Evaluation")
print("=" * 120)
print(ranked_best_pairs[available_cols].head(5).to_string(index=False))

print("\n" + "=" * 120)
print("ABSOLUTE WORST PAIRS (No Style Floor Imposed)")
print("Recommended for Section 5.2 Failure Mode Diagnosis & Saliency Heatmaps")
print("=" * 120)
print(ranked_worst_pairs[available_cols].head(5).to_string(index=False))

In [ ]:
"""Extract Top 5 Best and Worst Pairs
- Intra-architecture Style Normalization (-log10 Gram MSE)
- Qualifying Best: Style Fidelity > 10th percentile per architecture
- Unconstrained Worst: Full dataset to capture failure modes
- Displays summary tables
"""
# ==========================================
# 1. LOAD & SANITIZE DATASET
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets"
METRICS_CSV = os.path.join(
    BASE_DIR,
    "Output_subset/Comprehensive_Benchmark_Analysis/master_unified_metrics.csv",
)

df = pd.read_csv(METRICS_CSV)


def sanitize(val):
  s = str(val).lower().strip()
  for ext in [".jpg", ".jpeg", ".png", ".webp"]:
    s = s.replace(ext, "")
  if "starry" in s:
    return "starrynight"
  if "composition" in s:
    return "composition_vii"
  if "muse" in s:
    return "la_muse"
  if "feather" in s:
    return "feathers"
  return s.replace(" ", "_").replace("-", "_")


df["Content_Image"] = df["Content_Image"].apply(sanitize)
df["Style_Image"] = df["Style_Image"].apply(sanitize)

for col in ["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]:
  df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]
).copy()

# ==========================================
# 2. INTRA-ARCHITECTURE NORMALIZATION
# ==========================================
df["Score_Content"] = 0.5 * df["SSIM_Content"] + 0.5 * (1.0 - df["LPIPS_Content"])
df["Score_Style"] = 0.0
df["P10_Style"] = 0.0

for arch in df["Architecture"].unique():
  mask = df["Architecture"] == arch
  log_g = -np.log10(df.loc[mask, "Isolated_Gram_Style"] + 1e-12)
  min_g, max_g = log_g.min(), log_g.max()
  norm_style = (log_g - min_g) / (max_g - min_g + 1e-8)
  df.loc[mask, "Score_Style"] = norm_style
  df.loc[mask, "P10_Style"] = norm_style.quantile(0.10)

df["Balanced_Compound"] = 0.50 * df["Score_Content"] + 0.50 * df["Score_Style"]

arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN",
}
df["Arch_Tag"] = df["Architecture"].map(arch_map).fillna(df["Architecture"])

# ==========================================
# 3. EXTRACT TOP 5 BEST & WORST PAIRS
# ==========================================
# BEST: Score_Style > 10th percentile, pick highest Balanced_Compound per triad
qualifying_df = df[df["Score_Style"] > df["P10_Style"]].copy()
best_triads = (
    qualifying_df.sort_values(by="Balanced_Compound", ascending=False)
    .groupby(["Content_Image", "Style_Image", "Arch_Tag"], as_index=False)
    .first()
)

best_pivoted = best_triads.pivot(
    index=["Content_Image", "Style_Image"],
    columns="Arch_Tag",
    values=[
        "Output_Image",
        "Run_Index",
        "Balanced_Compound",
        "SSIM_Content",
        "LPIPS_Content",
        "Score_Style",
    ],
)
best_pivoted.columns = [
    f"{metric}_{arch}" for metric, arch in best_pivoted.columns
]
best_pivoted = best_pivoted.reset_index()

comp_cols = [
    c for c in best_pivoted.columns if c.startswith("Balanced_Compound_")
]
best_pivoted = best_pivoted.dropna(subset=comp_cols).copy()
best_pivoted["Pair_Mean_Compound"] = best_pivoted[comp_cols].mean(axis=1)
top_5_best = (
    best_pivoted.sort_values(by="Pair_Mean_Compound", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

# WORST: Unconstrained full dataset, pick lowest Balanced_Compound per triad
worst_triads = (
    df.sort_values(by="Balanced_Compound", ascending=True)
    .groupby(["Content_Image", "Style_Image", "Arch_Tag"], as_index=False)
    .first()
)

worst_pivoted = worst_triads.pivot(
    index=["Content_Image", "Style_Image"],
    columns="Arch_Tag",
    values=[
        "Output_Image",
        "Run_Index",
        "Balanced_Compound",
        "SSIM_Content",
        "LPIPS_Content",
        "Score_Style",
    ],
)
worst_pivoted.columns = [
    f"{metric}_{arch}" for metric, arch in worst_pivoted.columns
]
worst_pivoted = worst_pivoted.reset_index()

comp_cols_w = [
    c for c in worst_pivoted.columns if c.startswith("Balanced_Compound_")
]
worst_pivoted = worst_pivoted.dropna(subset=comp_cols_w).copy()
worst_pivoted["Pair_Mean_Compound"] = worst_pivoted[comp_cols_w].mean(axis=1)
top_5_worst = (
    worst_pivoted.sort_values(by="Pair_Mean_Compound", ascending=True)
    .head(5)
    .reset_index(drop=True)
)

# ==========================================
# 4. PRINT FORMATTED REPORT TABLES
# ==========================================
cols_summary = [
    "Content_Image",
    "Style_Image",
    "Pair_Mean_Compound",
    "Output_Image_Gatys",
    "Balanced_Compound_Gatys",
    "Output_Image_FastNST",
    "Balanced_Compound_FastNST",
    "Output_Image_AdaIN",
    "Balanced_Compound_AdaIN",
]
avail_cols_b = [c for c in cols_summary if c in top_5_best.columns]
avail_cols_w = [c for c in cols_summary if c in top_5_worst.columns]

print("=" * 125)
print(
    "TOP 5 BEST COMPOUND IMAGE PAIRS (Style > P10 Cutoff | Best Run per Model)"
)
print("Recommended for Section 5.3 Human Evaluation (MOS / 2AFC)")
print("=" * 125)
print(top_5_best[avail_cols_b].to_string(index=False))

print("\n" + "=" * 125)
print(
    "TOP 5 WORST COMPOUND IMAGE PAIRS (Unconstrained Failure Modes | Lowest"
    " Run)"
)
print("Recommended for Section 5.2 Failure Mode Analysis & Saliency Maps")
print("=" * 125)
print(top_5_worst[avail_cols_w].to_string(index=False))

In [ ]:
"""
Targeted Metric Extractor for Survey Benchmark Stimulus Triads
Extracts SSIM, LPIPS, Gram MSE, Content Scores, Style Scores, and Compound Utilities
for the exact 15 generated image runs selected for the human psychophysical survey.
"""

import os
import numpy as np
import pandas as pd

# ==========================================
# 1. LOAD MASTER DATASET
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Output_subset/Comprehensive_Benchmark_Analysis/master_unified_metrics.csv")
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not locate master evaluation CSV. Verify directory path.")

df = pd.read_csv(csv_path)

# ==========================================
# 2. NUMERIC CASTING & INTRA-ARCH NORMALIZATION
# ==========================================
df["SSIM_Content"] = pd.to_numeric(df["SSIM_Content"], errors="coerce")
df["LPIPS_Content"] = pd.to_numeric(df["LPIPS_Content"], errors="coerce")
df["Isolated_Gram_Style"] = pd.to_numeric(df["Isolated_Gram_Style"], errors="coerce")

df = df.dropna(subset=["SSIM_Content", "LPIPS_Content", "Isolated_Gram_Style"]).copy()

# Content metrics
df["Content_Score_Norm"] = 0.50 * df["SSIM_Content"] + 0.50 * (1.0 - df["LPIPS_Content"])
df["Content_Compound_Raw"] = df["SSIM_Content"] + (1.0 - df["LPIPS_Content"])

# Intra-architecture style normalization (-log10 Gram MSE)
df["Score_Style"] = 0.0
for arch in df["Architecture"].unique():
    mask = df["Architecture"] == arch
    log_g = -np.log10(df.loc[mask, "Isolated_Gram_Style"] + 1e-12)
    min_g, max_g = log_g.min(), log_g.max()
    df.loc[mask, "Score_Style"] = (log_g - min_g) / (max_g - min_g + 1e-8)

# 50/50 Balanced utility score
df["Balanced_Compound"] = 0.50 * df["Content_Score_Norm"] + 0.50 * df["Score_Style"]

# Standardize architecture labeling
arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN"
}
df["Arch_Tag"] = df["Architecture"].map(arch_map).fillna(df["Architecture"])

# ==========================================
# 3. FILTER TO THE EXACT 15 SURVEY RUNS
# ==========================================
survey_targets = [
    # (Pair Name, Arch_Tag, Target Filename)
    ("Angel x Wave", "Gatys", "angel1024_wave_run2.jpg"),
    ("Angel x Wave", "FastNST", "angel1024_wave_run2.jpg"),
    ("Angel x Wave", "AdaIN", "angel1024_wave_run1.jpg"),

    ("Mountain x Wave", "Gatys", "mountains1024_wave_run3.jpg"),
    ("Mountain x Wave", "FastNST", "mountains1024_wave_run3.jpg"),
    ("Mountain x Wave", "AdaIN", "mountains1024_wave_run3.jpg"),

    ("Berries x Wave", "Gatys", "berries1024_wave_run3.jpg"),
    ("Berries x Wave", "FastNST", "berries1024_wave_run1.jpg"),
    ("Berries x Wave", "AdaIN", "berries1024_wave_run1.jpg"),

    ("Berries x Composition", "Gatys", "berries1024_composition_vii_run2.jpg"),
    ("Berries x Composition", "FastNST", "berries1024_composition_vii_run3.jpg"),
    ("Berries x Composition", "AdaIN", "berries1024_composition_vii_run2.jpg"),

    ("Daisy x Wave", "Gatys", "daisy1024_wave_run2.jpg"),
    ("Daisy x Wave", "FastNST", "daisy1024_wave_run2.jpg"),
    ("Daisy x Wave", "AdaIN", "daisy1024_wave_run3.jpg"),
]

extracted_records = []

for pair_name, arch_tag, img_name in survey_targets:
    match = df[
        (df["Arch_Tag"] == arch_tag) &
        (df["Output_Image"].astype(str).str.contains(img_name, case=False, na=False))
    ]

    if match.empty:
        print(f"[!] Warning: Could not find exact match for {arch_tag} - {img_name}")
        continue

    row = match.iloc[0]
    extracted_records.append({
        "Stimulus_Pair": pair_name,
        "Architecture": arch_tag,
        "Output_Image": img_name,
        "SSIM": round(float(row["SSIM_Content"]), 4),
        "LPIPS": round(float(row["LPIPS_Content"]), 4),
        "Score_Content_[0-1]": round(float(row["Content_Score_Norm"]), 4),
        "Content_Raw_[0-2]": round(float(row["Content_Compound_Raw"]), 4),
        "Isolated_Gram_Style": f"{row['Isolated_Gram_Style']:.2e}",
        "Score_Style_[0-1]": round(float(row["Score_Style"]), 4),
        "Balanced_Compound": round(float(row["Balanced_Compound"]), 4),
    })

survey_df = pd.DataFrame(extracted_records)

# ==========================================
# 4. DISPLAY SUMMARY TABLE
# ==========================================
print("=" * 145)
print("EXACT METRICS FOR THE 15 HUMAN EVALUATION BENCHMARK IMAGES")
print("=" * 145)
display_cols = [
    "Stimulus_Pair", "Architecture", "Output_Image",
    "SSIM", "LPIPS", "Score_Content_[0-1]", "Content_Raw_[0-2]",
    "Isolated_Gram_Style", "Score_Style_[0-1]", "Balanced_Compound"
]
print(survey_df[display_cols].to_string(index=False))

In [ ]:
"""
Extract Style Gram Loss for Human Survey Images
Extracts raw Gram MSE for the 15 benchmark stimulus images, formatting both as
standard scientific notation (e.g., 3.4512e-04) and scaled by 10^-4 (x 10^4).
"""
# ==========================================
# 1. LOAD MASTER METRICS DATASET
# ==========================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets"
POSSIBLE_PATHS = [
    os.path.join(BASE_DIR, "Output_subset/Comprehensive_Benchmark_Analysis/master_unified_metrics.csv")
]

csv_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
if not csv_path:
    raise FileNotFoundError("Could not find master_unified_metrics.csv. Check directory path.")

df = pd.read_csv(csv_path)

# Ensure numeric casting
df["Isolated_Gram_Style"] = pd.to_numeric(df["Isolated_Gram_Style"], errors="coerce")

# Standardize architecture tag
arch_map = {
    "Gatys (Optimization)": "Gatys",
    "Fast-NST (Johnson)": "FastNST",
    "AdaIN (Huang)": "AdaIN",
}
df["Arch_Tag"] = df["Architecture"].map(arch_map).fillna(df["Architecture"])

# ==========================================
# 2. DEFINE THE 15 SURVEY BENCHMARK RUNS
# ==========================================
survey_targets = [
    ("Angel x Wave", "Gatys", "angel1024_wave_run2.jpg"),
    ("Angel x Wave", "FastNST", "angel1024_wave_run2.jpg"),
    ("Angel x Wave", "AdaIN", "angel1024_wave_run1.jpg"),

    ("Daisy x Wave", "Gatys", "daisy1024_wave_run2.jpg"),
    ("Daisy x Wave", "FastNST", "daisy1024_wave_run2.jpg"),
    ("Daisy x Wave", "AdaIN", "daisy1024_wave_run3.jpg"),

    ("Mountain x Wave", "Gatys", "mountains1024_wave_run3.jpg"),
    ("Mountain x Wave", "FastNST", "mountains1024_wave_run3.jpg"),
    ("Mountain x Wave", "AdaIN", "mountains1024_wave_run3.jpg"),

    ("Berries x Wave", "Gatys", "berries1024_wave_run3.jpg"),
    ("Berries x Wave", "FastNST", "berries1024_wave_run1.jpg"),
    ("Berries x Wave", "AdaIN", "berries1024_wave_run1.jpg"),

    ("Berries x Composition", "Gatys", "berries1024_composition_vii_run2.jpg"),
    ("Berries x Composition", "FastNST", "berries1024_composition_vii_run3.jpg"),
    ("Berries x Composition", "AdaIN", "berries1024_composition_vii_run2.jpg"),
]

# ==========================================
# 3. EXTRACT AND FORMAT GRAM LOSS
# ==========================================
records = []

for pair_name, arch_tag, img_name in survey_targets:
    match = df[
        (df["Arch_Tag"] == arch_tag) &
        (df["Output_Image"].astype(str).str.contains(img_name, case=False, na=False))
    ]

    if match.empty:
        print(f"[!] Target not found: {arch_tag} - {img_name}")
        continue

    row = match.iloc[0]
    gram_val = float(row["Isolated_Gram_Style"])

    records.append({
        "Stimulus_Pair": pair_name,
        "Architecture": arch_tag,
        "Output_Image": img_name,
        "Gram_Raw": gram_val,
        # Scientific notation with 4 decimals (e.g., 2.3415e-04)
        "Gram_Scientific_e-04": f"{gram_val:.4e}",
        # Scaled in units of 10^-4 (value * 10,000)
        "Gram_x_10^4": round(gram_val * 1e4, 4),
        # Direct 4 decimal precision
        "Gram_4_Decimals": round(gram_val, 4),
    })

gram_df = pd.DataFrame(records)

# ==========================================
# 4. PRINT FORMATTED TABLE
# ==========================================
print("=" * 115)
print("EXTRACTED STYLE GRAM LOSS FOR SURVEY IMAGES")
print("=" * 115)
print(
    gram_df[
        [
            "Stimulus_Pair",
            "Architecture",
            "Output_Image",
            "Gram_Scientific_e-04",
            "Gram_x_10^4",
            "Gram_4_Decimals",
        ]
    ].to_string(index=False)
)

In [ ]:
"""
Targeted Fast-NST Saliency Map Extractor (Restricted to 'nprgeneral' Content Source)
Extracts conv4_2 gradient saliency maps for missing:
  - angel1024_wave_run2.jpg
  - rim1024_mosaic_run3.jpg
Sourcing content images strictly from the 'nprgeneral' directory.
Exports standalone borderless 'magma' heatmaps (.png) and NumPy matrices (.npy).
"""

from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

# ==============================================================================
# 1. HARDWARE SELECTION & VGG-19 conv4_2 SLICE
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

class VGG19ContentSaliency(nn.Module):
    def __init__(self):
        super(VGG19ContentSaliency, self).__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features
        # Layer 21 corresponds to conv4_2 in torchvision VGG-19 features
        self.features = nn.Sequential(*[vgg[i] for i in range(22)]).to(device).eval()
        for param in self.features.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.features(x)

vgg_model = VGG19ContentSaliency()

# ==============================================================================
# 2. ISOLATED SALIENCY EXTRACTION
# ==============================================================================
def compute_isolated_saliency(content_path, stylized_path):
    c_raw = Image.open(content_path).convert("RGB")
    s_raw = Image.open(stylized_path).convert("RGB")

    c_tensor = preprocess(c_raw).unsqueeze(0).to(device)
    s_tensor = preprocess(s_raw).unsqueeze(0).to(device)

    s_tensor.requires_grad_(True)

    feat_c = vgg_model(c_tensor)
    feat_s = vgg_model(s_tensor)

    loss = 0.5 * torch.sum((feat_s - feat_c) ** 2)
    loss.backward()

    # Channel-wise max absolute gradient magnitude
    grad = s_tensor.grad.detach().cpu().squeeze(0).abs()
    saliency, _ = torch.max(grad, dim=0)
    saliency = saliency.numpy()

    # Percentile clipping (1st to 99th) to normalize dynamic range to [0, 1]
    p99 = np.percentile(saliency, 99)
    p01 = np.percentile(saliency, 1)
    saliency = np.clip((saliency - p01) / (p99 - p01 + 1e-8), 0.0, 1.0)

    return saliency

# ==============================================================================
# 3. DIRECTORY RESOLUTION (STRICT 'nprgeneral' CONTENT FILTER)
# ==============================================================================
BASE_DIR = "/content/drive/MyDrive/NST_Datasets"

def find_nprgeneral_folder(root_dir):
    """Locates the directory containing 'nprgeneral' anywhere under root."""
    for root, dirs, _ in os.walk(root_dir):
        for d in dirs:
            if "nprgeneral" in d.lower():
                return os.path.join(root, d)
    return None

def find_file_in_dir(directory, token):
    """Finds an image file inside a specific directory matching a keyword token."""
    for root, _, files in os.walk(directory):
        for f in files:
            name = f.lower()
            if token.lower() in name and any(name.endswith(ext) for ext in [".jpg", ".jpeg", ".png"]):
                return os.path.join(root, f)
    return None

def find_exact_file(filename, search_root):
    """Walks the directory to locate an exact filename match."""
    for root, _, files in os.walk(search_root):
        for f in files:
            if f.lower() == filename.lower():
                return os.path.join(root, f)
    return None

# Resolve nprgeneral directory
nprgeneral_dir = find_nprgeneral_folder(BASE_DIR)
if not nprgeneral_dir:
    # Fallback to direct path search in case it's in the root
    if os.path.exists(os.path.join(BASE_DIR, "nprgeneral")):
        nprgeneral_dir = os.path.join(BASE_DIR, "nprgeneral")
    else:
        raise FileNotFoundError(f"Could not locate folder 'nprgeneral' under {BASE_DIR}")

print(f"[✓] Locked content source directory: {nprgeneral_dir}\n")

TARGETS = [
    {
        "content_token": "angel",
        "stylized_file": "angel1024_wave_run2.jpg",
        "out_name": "saliency_FastNST_angel_wave_run2"
    },
    {
        "content_token": "rim",
        "stylized_file": "rim1024_mosaic_run3.jpg",
        "out_name": "saliency_FastNST_rim_mosaic_run3"
    }
]

OUTPUT_DIR = os.path.join(BASE_DIR, "Output_subset/Comprehensive_Benchmark_Analysis/Saliency_Maps")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 4. EXECUTE EXTRACTION
# ==============================================================================
print("=" * 80)
print("EXTRACTING FAST-NST SALIENCY MAPS (SOURCE: nprgeneral)")
print("=" * 80)

for item in TARGETS:
    target_img_name = item["stylized_file"]
    token = item["content_token"]
    label = item["out_name"]

    # 1. Stylized image from FastNST directory
    stylized_path = find_exact_file(target_img_name, BASE_DIR)

    # 2. Content image strictly from nprgeneral
    content_path = find_file_in_dir(nprgeneral_dir, token)

    if not stylized_path:
        print(f"[!] Stylized file not found: {target_img_name}")
        continue
    if not content_path:
        print(f"[!] Content token '{token}' not found inside {nprgeneral_dir}")
        continue

    print(f"\n[+] Processing: {label}")
    print(f"    Content (nprgeneral) : {content_path}")
    print(f"    FastNST Output       : {stylized_path}")

    # Generate gradient saliency
    saliency = compute_isolated_saliency(content_path, stylized_path)

    # Save borderless heatmap image
    png_path = os.path.join(OUTPUT_DIR, f"{label}.png")
    plt.imsave(png_path, saliency, cmap="magma")

    # Save raw array matrix
    npy_path = os.path.join(OUTPUT_DIR, f"{label}.npy")
    np.save(npy_path, saliency)

    print(f"    [✓] Heatmap: {png_path}")
    print(f"    [✓] Matrix : {npy_path}")

print("\n" + "=" * 80)
print("Extraction complete.")